In [1]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Running locally (not in Colab)")

import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Set project paths
PROJECT_ROOT = '/content/drive/MyDrive/Colab_Projects/BigData/' if IN_COLAB else './'
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/models', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/results', exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"Colab Environment: {IN_COLAB}")

Mounted at /content/drive
Project Root: /content/drive/MyDrive/Colab_Projects/BigData/
Colab Environment: True


In [2]:
# Install required packages for Colab
import subprocess

packages_to_install = [
    'pyspark',
    'shap',
    'lime',
    'gradio',
    'tensorflow',
    'scikit-learn',
    'pandas',
    'numpy',
    'matplotlib',
    'seaborn',
    'plotly'
]

for package in packages_to_install:
    try:
        __import__(package)
        print(f"✓ {package} already installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("\nAll packages installed successfully!")

✓ pyspark already installed
✓ shap already installed
Installing lime...
✓ gradio already installed
✓ tensorflow already installed
Installing scikit-learn...
✓ pandas already installed
✓ numpy already installed
✓ matplotlib already installed
✓ seaborn already installed
✓ plotly already installed

All packages installed successfully!


# Initialize PySpark with optimized Colab configuration
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, MultilayerPerceptronClassifier

# Spark Configuration for Colab
spark = SparkSession.builder \
    .appName("BigDataAnomalyDetection") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .config("spark.rdd.compress", "true") \
    .config("spark.shuffle.compress", "true") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"Spark App: {spark.sparkContext.appName}")
print(f"Spark Configuration:")
for key, value in spark.sparkContext.getConf().getAll():
    if 'memory' in key.lower() or 'partition' in key.lower():
        print(f"  {key}: {value}")

import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ARCHITECTURE DOCUMENTATION
architecture_doc = """
=== DISTRIBUTED SYSTEM ARCHITECTURE ===

1. DATA PARTITIONING STRATEGY:
   - Primary: Temporal partitioning (by hour/day)
   - Secondary: Hash partitioning by source_ip (200 partitions)
   - Rationale: Enables efficient time-series queries and parallel processing of IP traffic

2. CLUSTER CONFIGURATION (Colab-Optimized):
   - Driver Memory: 8GB (max safe for Colab)
   - Shuffle Partitions: 200 (balanced for 1M records)
   - Default Parallelism: 200 (matches shuffle partitions)
   - Adaptive Query Execution: Enabled (auto-tunes partitions)

3. DATA VARIETY SPECIFICATION:
   - Structured: Network metadata (timestamp, IPs, ports, protocols)
   - Numerical: Packet counts, bytes transferred, duration
   - Categorical: Protocol types, flags, anomaly labels
   - Temporal: Time-based patterns and behaviors

4. SCALABILITY JUSTIFICATION:
   - Lazy evaluation prevents memory overflow
   - Partition-based processing distributes computation
   - No .collect() on full dataset (only aggregated results)
"""

print(architecture_doc)
with open(f'{PROJECT_ROOT}/ARCHITECTURE.md', 'w') as f:
    f.write(architecture_doc)